In [1]:
from __future__ import print_function, absolute_import
import argparse
import os.path as osp
import random
import sys
import time
from datetime import timedelta

import numpy as np
import torch
from torch.utils.data import DataLoader

from pcr import datasets
from pcr.models.bpbreid_encoder import BPBReIDEncoder, BPBReIDModelCfg
from pcr.models.clip_text_encoder import ClipTextEncoder
from pcr.models.prompt_learner import PromptLearner
from pcr.models.relation_blocks import VisualAttentionBlock
from pcr.loss.clip_supcon_loss import SupConLoss
from pcr.utils.config import load_yaml_config
from pcr.utils.data import transforms as T
from pcr.utils.data.preprocessor import Preprocessor
from pcr.utils.logging import Logger
from pcr.utils.lr_scheduler import WarmupCosineLR
from pcr.utils.osutils import mkdir_if_missing
from pcr.utils.visibility_filter import filter_by_visibility


In [2]:
def get_data(name, data_dir):
    return datasets.create(name, osp.join(data_dir, name))


def get_cache_loader(dataset_list, root, height, width, batch_size, workers):
    normalizer = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    transformer = T.Compose([
        T.Resize((height, width), interpolation=3),
        T.ToTensor(),
        normalizer,
    ])
    return DataLoader(
        Preprocessor(dataset_list, root=root, transform=transformer),
        batch_size=batch_size, num_workers=workers, shuffle=False, pin_memory=True)

def cache_part_features(encoder, data_loader):
    """Single full-dataset forward pass under no_grad, caching every image's part embeddings,
    visibility, and real identity label -- mirrors CLIP-ReID's own stage-1 full-dataset feature
    cache, generalized to BPBreID's [M, D] per-branch embeddings."""
    encoder.eval()
    features, visibilities, labels = [], [], []
    with torch.no_grad():
        for imgs, _, pids, _, _ in data_loader:
            f_out, vis = encoder(imgs.cuda())
            features.append(f_out.cpu())
            visibilities.append(vis.cpu())
            labels.append(pids)
    return torch.cat(features, 0), torch.cat(visibilities, 0), torch.cat(labels, 0)


def build_encoder(cfg):
    model_cfg = BPBReIDModelCfg(backbone=cfg.model.backbone)
    model_cfg.masks.parts_num = cfg.model.parts_num
    model_cfg.dim_reduce_output = cfg.model.dim_reduce_output
    encoder = BPBReIDEncoder(model_cfg, checkpoint_path=cfg.model.checkpoint_path or None).cuda()
    encoder.eval()
    for p in encoder.parameters():
        p.requires_grad_(False)
    return encoder

In [3]:
cfg = load_yaml_config("configs/stage1_relational_prompts.yaml")

In [5]:
cfg.data.batch_size = 4

In [6]:
dataset = get_data(cfg.data.dataset, cfg.data.data_dir)
num_identities = dataset.num_train_pids
num_parts = cfg.model.parts_num
num_branches = 1 + num_parts

=> Market1501 loaded
Dataset statistics:
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |   751 |    12936 |         6
  query    |   750 |     3368 |         6
  gallery  |   751 |    15913 |         6
  ----------------------------------------


In [8]:
encoder = build_encoder(cfg)
text_encoder = ClipTextEncoder(clip_arch=cfg.clip.arch, device='cuda').cuda()
prompt_learner = PromptLearner(num_identities, num_parts, text_encoder, n_ctx=cfg.clip.n_ctx,
                                tab_num_heads=cfg.tab.num_heads, tab_num_layers=cfg.tab.num_layers,
                                device='cuda').cuda()
vab = VisualAttentionBlock(dim=cfg.model.dim_reduce_output, num_heads=cfg.vab.num_heads,
                            num_layers=cfg.vab.num_layers).cuda()

=> init weights from normal distribution
Successfully loaded pretrained weights from "examples/logs/stage0_bpa/model_best.pth.tar"


/home/lakshh/workspace/reid/pcr2/pcr/models/relation_blocks.py:75: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
/home/lakshh/workspace/reid/pcr2/pcr/models/relation_blocks.py:44: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)


In [11]:
train_set, _ = filter_by_visibility(
    sorted(dataset.train), encoder, cfg.data.height, cfg.data.width,
    cfg.visibility.lambda_v_min, root=dataset.images_dir, batch_size=cfg.data.cache_batch_size,
    workers=cfg.data.workers)
    
cache_loader = get_cache_loader(train_set, dataset.images_dir, cfg.data.height, cfg.data.width,
    cfg.data.cache_batch_size, cfg.data.workers)

==> Visibility filter (threshold=0.50): kept 12824/12936 images, rejected 112


In [ ]:
cached_features, cached_visibility, cached_labels = cache_part_features(encoder, cache_loader)
cached_features = cached_features.cuda()
cached_labels = cached_labels.cuda()
num_images = cached_labels.size(0)
print("==> Cached {} images across {} identities, {} branches".format(
    num_images, num_identities, num_branches))

==> Cached 12824 images across 751 identities, 6 branches


In [13]:
cached_features.shape

torch.Size([12824, 6, 512])

In [15]:
from pcr.loss.clip_infonce_loss import InfoNCELoss

infonce = InfoNCELoss(temperature=cfg.loss.temperature).cuda()
# fg_ctx deliberately excluded -- Algorithm 1 has no foreground term (see module docstring);
# only part_ctx, TextualAttentionBlock (prompt_learner.tab), and VisualAttentionBlock train.
trainable_params = ([prompt_learner.part_ctx] + list(prompt_learner.tab.parameters())
                        + list(vab.parameters()))
optimizer = torch.optim.Adam(trainable_params, lr=cfg.optim.lr,
                                weight_decay=cfg.optim.weight_decay)
scheduler = WarmupCosineLR(optimizer, max_epochs=cfg.optim.epochs,
                            warmup_epochs=cfg.optim.warmup_epochs,
                            warmup_lr_init=cfg.optim.warmup_lr_init,
                            lr_min=cfg.optim.lr_min)
# GradScaler, not raw fp16 backward -- the CLIP text tower runs in fp16 (matches CLIP-ReID's
# own dtype exactly, see pcr/models/clip_text_encoder.py's docstring), and CLIP-ReID's own
# stage-1 loop always wraps its backward in a GradScaler to guard against fp16 gradient
# underflow through the text transformer -- ported faithfully rather than assuming raw fp16
# backward is fine. VisualAttentionBlock and PromptLearner's own parameters run in fp32
# (GradScaler is harmless for fp32 leaves), so one scaler covers everything trainable.
scaler = torch.amp.GradScaler('cuda')

In [16]:
def build_pk_batches(cached_labels, num_instances, batch_size):
    """Groups the cached feature set's indices by identity, then partitions all identities into
    PK batches for one epoch: batch_size // num_instances identities per batch, num_instances
    cached images per identity (sampled with replacement if that identity has fewer than
    num_instances cached images). Algorithm 1 step 6 ("Sample a PK batch of pre-filtered
    images") -- see this file's own module docstring for why this matters even with InfoNCELoss's
    per-identity deduplication making it safe against PK-batch collisions. A final partial group
    of identities (fewer than batch_size // num_instances left over) is dropped, matching this
    repo's other PK
    samplers' drop_last convention (pcr/utils/data/sampler.py::RandomIdentitySampler)."""
    labels_np = cached_labels.cpu().numpy()
    id_to_indices = {}
    for idx, pid in enumerate(labels_np):
        id_to_indices.setdefault(int(pid), []).append(idx)
    pids = list(id_to_indices.keys())
    random.shuffle(pids)

    num_pids_per_batch = max(1, batch_size // num_instances)
    batches = []
    for start in range(0, len(pids), num_pids_per_batch):
        batch_pids = pids[start:start + num_pids_per_batch]
        if len(batch_pids) < num_pids_per_batch:
            break
        batch_idx = []
        for pid in batch_pids:
            pool = id_to_indices[pid]
            replace = len(pool) < num_instances
            chosen = np.random.choice(pool, size=num_instances, replace=replace)
            batch_idx.extend(int(i) for i in chosen)
        batches.append(torch.tensor(batch_idx, dtype=torch.long, device=cached_labels.device))
    return batches

prompt_learner.train()
vab.train()
epoch_loss = 0.0
epoch_start = time.time()
# Algorithm 1 step 6: a fresh PK partition of the cached feature set every epoch, not a
# plain random sub-batch -- see build_pk_batches' and this file's own module docstring.
batches = build_pk_batches(cached_labels, cfg.data.num_instances, cfg.data.batch_size)
iters_per_epoch = len(batches)

In [17]:
b_idx = batches[0]
b_labels = cached_labels[b_idx]
b_features = cached_features[b_idx]  # [b, 1+K, D], already L2-normalized per branch

optimizer.zero_grad()

# Algorithm 1 steps 10-14: only the K part prompts are built and pushed through the
# frozen CLIP text encoder -- no foreground prompt/loss (module docstring).
prompts = prompt_learner.build_part_prompts(b_labels)  # list of 1+K tensors
part_visual = vab(b_features[:, 1:, :])  # [b, K, D], relationally mixed
loss = b_features.new_zeros(())

In [21]:
len(prompts[0])
prompts[0].shape

torch.Size([4, 77, 512])

In [22]:
part_visual = vab(b_features[:, 1:, :])

In [ ]:
part_visual[0].shape

torch.Size([5, 512])

: 

# Stage 0 ablation: what is each part actually looking at?

Stage 0 trains BPBreID's pixel-to-part classifier (BPAM) to split every pixel of a person photo
into one of 6 classes: `background`, and 5 body parts (`part_1..part_5`). This section loads the
real, fully-trained (60-epoch) Stage 0 checkpoint, runs it on a handful of **random** training
images, and visualizes each part's own attention map directly on top of the photo -- a simple,
qualitative way to check whether each part branch settled on a distinct, sensible body region, or
collapsed onto something degenerate (a real failure mode this repo hit earlier with an
under-trained, 1-epoch checkpoint -- see `progress.md`).

Self-contained: builds its own encoder from `examples/logs/stage0_bpa/model_best.pth.tar` directly
(HRNet32, 512-dim, matching that checkpoint's own saved shapes), independent of whatever `cfg`/
`encoder` the notebook's earlier cells left behind.

In [ ]:
import os.path as osp
import random

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms as tv_transforms

from pcr import datasets
from pcr.models.bpbreid_encoder import BPBReIDEncoder, BPBReIDModelCfg

# Stage 0's own real checkpoint -- HRNet32, dim_reduce_output=512, trained the full 60 epochs
# (confirmed directly: checkpoint['epoch'] == 60), not one of the 1-epoch smoke-test artifacts
# from earlier in this repo's history.
STAGE0_CHECKPOINT = 'examples/logs/stage0_bpa/model_best.pth.tar'
HEIGHT, WIDTH = 384, 128
NUM_PARTS = 5
NUM_SAMPLES = 6
SEED = 7

stage0_model_cfg = BPBReIDModelCfg(backbone='hrnet32')
stage0_model_cfg.masks.parts_num = NUM_PARTS
stage0_model_cfg.dim_reduce_output = 512
# Continuous (not the dataclass's own binary default) -- matches Stage 1/2's own build_encoder,
# and is far more informative for this kind of qualitative check than a collapsed-to-{0,1}
# indicator would be.
stage0_model_cfg.training_binary_visibility_score = False
stage0_model_cfg.testing_binary_visibility_score = False

stage0_encoder = BPBReIDEncoder(stage0_model_cfg, checkpoint_path=STAGE0_CHECKPOINT).cuda()
stage0_encoder.eval()
for p in stage0_encoder.parameters():
    p.requires_grad_(False)

In [ ]:
# Random person images -- re-picking freshly from dataset.train each run (change SEED to look at
# a different set of people).
ablation_dataset = datasets.create('market1501', osp.join('../datasets', 'market1501'))

random.seed(SEED)
samples = random.sample(ablation_dataset.train, NUM_SAMPLES)

normalizer = tv_transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
to_model = tv_transforms.Compose([
    tv_transforms.Resize((HEIGHT, WIDTH), interpolation=tv_transforms.InterpolationMode.BICUBIC),
    tv_transforms.ToTensor(),
    normalizer,
])

raw_images, model_inputs = [], []
for fname, pid, camid in samples:
    fpath = fname if ablation_dataset.images_dir is None else osp.join(ablation_dataset.images_dir, fname)
    img = Image.open(fpath).convert('RGB')
    raw_images.append(np.array(img.resize((WIDTH, HEIGHT))))
    model_inputs.append(to_model(img))

batch = torch.stack(model_inputs, dim=0).cuda()

with torch.no_grad():
    f_out, vis, pixels_cls_scores = stage0_encoder.forward_full(batch)
    # pixels_cls_scores: [B, 1+K, Hf, Wf] raw per-pixel logits over {background, part_1..part_K}
    pixel_probs = F.softmax(pixels_cls_scores, dim=1)
    pixel_probs_up = F.interpolate(pixel_probs, size=(HEIGHT, WIDTH), mode='bilinear', align_corners=True)

pixel_probs_up = pixel_probs_up.cpu().numpy()  # [B, 1+K, H, W]
vis = vis.cpu().numpy()  # [B, 1+K]
part_names = ['background'] + [f'part_{k+1}' for k in range(NUM_PARTS)]

In [ ]:
# Quantitative check first: is any branch degenerate (stuck near 0 or near 1 for every image,
# or soaking up spatial mass that should belong to its neighbors)?
print("Per-branch visibility score (mean over {} sample images):".format(NUM_SAMPLES))
for b, name in enumerate(part_names):
    print(f"  {name:12s}  mean={vis[:, b].mean():.3f}  min={vis[:, b].min():.3f}  max={vis[:, b].max():.3f}")

print()
print("Per-branch spatial attention mass (mean softmax probability per pixel, averaged over all")
print("pixels and all sample images -- 1/6=0.167 is the 'uniform, learned nothing' baseline):")
for b, name in enumerate(part_names):
    print(f"  {name:12s}  mean_prob={pixel_probs_up[:, b].mean():.4f}  max_prob={pixel_probs_up[:, b].max():.4f}")

In [ ]:
# Now look at it directly: original image + each branch's attention heatmap, one row per person.
fig, axes = plt.subplots(NUM_SAMPLES, 1 + len(part_names),
                          figsize=(2.2 * (1 + len(part_names)), 2.6 * NUM_SAMPLES))
for i in range(NUM_SAMPLES):
    axes[i, 0].imshow(raw_images[i])
    axes[i, 0].set_title('image' if i == 0 else '', fontsize=9)
    axes[i, 0].axis('off')
    for b, name in enumerate(part_names):
        ax = axes[i, 1 + b]
        ax.imshow(raw_images[i])
        ax.imshow(pixel_probs_up[i, b], cmap='jet', alpha=0.55, vmin=0, vmax=1)
        if i == 0:
            ax.set_title(name, fontsize=9)
        ax.axis('off')

plt.tight_layout()
plt.show()

## What this actually shows (ran with `SEED = 7`)

**Numbers.** Every branch's visibility score sits in the 0.90-0.95 range on average, except
`part_1` (0.77 average, but ranging all the way down to 0.04 for one sample) -- that's `part_1`
correctly reporting "I can't find myself" on the one photo where the person's head happens to be
out of frame or heavily occluded, not a branch that's permanently broken. No branch sits pinned at
exactly 0 or 1 across every image, and no branch's spatial attention mass swallows the whole image
(the spread is `part_1`/`part_5` around 0.05, `part_2`-`part_4` around 0.16-0.18, `background`
around 0.39) -- a real, varied distribution, not a collapse.

**Picture.** Reading left to right across each row: `part_1` consistently lands on the **head**,
`part_2` on the **shoulders/chest**, `part_3` on the **waist/hips**, `part_4` on the **thighs**,
and `part_5` on the **feet/ankles** -- a clean top-to-bottom split of the body that holds up across
6 different people, poses, clothing colors, and even a partially-occluded pair (row 5) and a person
with their head barely in frame (row 6). `background` correctly lights up the scene around the
person's silhouette, not the person themselves.

**Why this is worth checking at all**: an earlier point in this repo's history diagnosed the
*opposite* pattern from a 1-epoch smoke-tested checkpoint -- one branch reading as permanently
invisible across the entire dataset, another reading as visible ~96% of the time regardless of
image content (see `progress.md`'s entries on the visibility-weighting refactor). That was the
signature of an attention head that hadn't learned anything yet. This checkpoint (`epoch: 60`,
confirmed by loading it directly) shows none of that -- it's a working, anatomically-sensible part
decomposition. This is the checkpoint Stage 1/2 should be building on.

(Re-run the previous three cells with a different `SEED` to sanity-check this holds for other
random people too, not just this particular sample of 6.)

# VAB/TAB ablation: is there an attention signal, and can it be seen "in image space"?

Short answer up front: **yes for VAB, only partly for TAB.**

VAB and TAB are both tiny `nn.TransformerEncoder`s, but over a completely different kind of token
than Stage 0's pixel classifier:

- **VisualAttentionBlock (VAB)** runs self-attention over **K=5 tokens**, one per body part --
  each token is that part's *already pooled* embedding (one vector, no spatial grid left at all).
  Its attention is therefore a small **5x5 matrix** ("how much does output part *i* draw from
  input part *j*"), not a pixel-level map. But because each input token still traces back to a
  real image region (Stage 0's own part map for that part), that 5x5 matrix *can* be projected
  back onto the photo: for output token *i*, weight each of the 5 original part maps by how much
  attention *i* actually gave it, and add them up. That gives a genuine "which pixels actually
  influenced this output token" heatmap -- the closest real equivalent of an image-space attention
  map VAB has.

- **TextualAttentionBlock (TAB)** runs self-attention over the **learnable text-prompt context**
  for one identity -- pure parameters, indexed by identity, that never touch an image at all (see
  `pcr/models/prompt_learner.py`). It has a real attention matrix too (K*n_ctx x K*n_ctx, shown
  below aggregated down to 5x5 for readability), but there is **no image behind any of its
  tokens** -- so it can only ever be shown as a part-to-part matrix, never as a photo overlay.
  That's a property of what TAB actually is, not a limitation of the visualization.

Both `nn.TransformerEncoderLayer`'s own forward pass normally throws its attention weights away
(`need_weights=False`, hardcoded) -- the cells below pull them out by manually replaying one
layer's own `norm1` + `self_attn` call with `need_weights=True`, using that block's real, loaded
weights. Uses the most complete matching Stage 1 artifact set available on this machine
(`examples/logs/smoke_stage1_widepool/`, paired with its own Stage 0 source checkpoint) -- **only
2 epochs of Stage 1 training**, much shorter than the 60-epoch Stage 0 checkpoint used above, so
treat what follows as "does the technique work and what does early training look like", not a
verdict on the final design.

In [ ]:
from pcr.models.relation_blocks import VisualAttentionBlock, _visibility_attn_bias
from pcr.models.prompt_learner import PromptLearner
from pcr.models.clip_text_encoder import ClipTextEncoder
from pcr.utils.serialization import load_checkpoint

PROMPT_DIR = 'examples/logs/smoke_stage1_widepool'
VAB_STAGE0_CHECKPOINT = 'examples/logs/smoke_stage0_1024/model_best.pth.tar'  # matches this VAB's own training
VAB_BACKBONE = 'resnet50'
VAB_DIM = 1024
N_CTX = 4
CLIP_ARCH = 'RN50'
NUM_ATTN_SAMPLES = 4
ATTN_SEED = 3

# Self-contained, like the Stage 0 section above -- doesn't rely on `dataset`/`num_identities`
# from the notebook's earlier (crashed) cells.
attn_dataset = datasets.create('market1501', osp.join('../datasets', 'market1501'))
num_identities_attn = attn_dataset.num_train_pids

vab_model_cfg = BPBReIDModelCfg(backbone=VAB_BACKBONE)
vab_model_cfg.masks.parts_num = NUM_PARTS
vab_model_cfg.dim_reduce_output = VAB_DIM
vab_model_cfg.training_binary_visibility_score = False
vab_model_cfg.testing_binary_visibility_score = False
vab_encoder = BPBReIDEncoder(vab_model_cfg, checkpoint_path=VAB_STAGE0_CHECKPOINT).cuda()
vab_encoder.eval()
for p in vab_encoder.parameters():
    p.requires_grad_(False)

vab = VisualAttentionBlock(dim=VAB_DIM, num_heads=4, num_layers=1).cuda()
vab.load_state_dict(load_checkpoint(osp.join(PROMPT_DIR, 'vab.pth')))
vab.eval()

vab_text_encoder = ClipTextEncoder(clip_arch=CLIP_ARCH, device='cuda').cuda()
prompt_learner = PromptLearner(num_identities_attn, NUM_PARTS, vab_text_encoder, n_ctx=N_CTX,
                                tab_num_heads=4, tab_num_layers=1, device='cuda').cuda()
prompt_learner.load_state_dict(load_checkpoint(osp.join(PROMPT_DIR, 'prompt_learner.pth')))
prompt_learner.eval()

attn_identity_visibility = load_checkpoint(osp.join(PROMPT_DIR, 'identity_visibility.pth')).cuda()

In [ ]:
random.seed(ATTN_SEED)
attn_samples = random.sample(attn_dataset.train, NUM_ATTN_SAMPLES)

attn_raw_images, attn_model_inputs, attn_pids = [], [], []
for fname, pid, camid in attn_samples:
    fpath = fname if attn_dataset.images_dir is None else osp.join(attn_dataset.images_dir, fname)
    img = Image.open(fpath).convert('RGB')
    attn_raw_images.append(np.array(img.resize((WIDTH, HEIGHT))))
    attn_model_inputs.append(to_model(img))
    attn_pids.append(pid)
attn_batch = torch.stack(attn_model_inputs, dim=0).cuda()
attn_pids_t = torch.tensor(attn_pids, device='cuda')

with torch.no_grad():
    attn_f_out, attn_vis, attn_pixels_cls_scores = vab_encoder.forward_full(attn_batch)
    attn_pixel_probs = F.softmax(attn_pixels_cls_scores, dim=1)
    attn_pixel_probs_up = F.interpolate(attn_pixel_probs, size=(HEIGHT, WIDTH), mode='bilinear',
                                         align_corners=True).cpu().numpy()

    # ---- VAB attention: manually replay what VisualAttentionBlock.forward does internally, but
    # ask self_attn for the weights it normally throws away (need_weights=False, hardcoded). ----
    part_tokens = attn_f_out[:, 1:, :]           # [B, K, D]
    part_visibility = attn_vis[:, 1:]            # [B, K]
    vab_bias = _visibility_attn_bias(part_visibility, vab.num_heads)
    vab_layer = vab.encoder.layers[0]
    vab_normed = vab_layer.norm1(part_tokens)    # norm_first=True, same as the real forward pass
    _, vab_attn = vab_layer.self_attn(vab_normed, vab_normed, vab_normed, attn_mask=vab_bias,
                                       need_weights=True, average_attn_weights=False)
    vab_attn = vab_attn.mean(dim=1).cpu().numpy()  # [B, num_heads, K, K] -> [B, K, K], averaged over heads

    # ---- TAB attention: same technique, on this identity's own learnable prompt context ----
    id_vis = attn_identity_visibility[attn_pids_t]
    raw_part_ctx = prompt_learner.part_ctx[attn_pids_t]              # [B, K*n_ctx, ctx_dim]
    token_vis = id_vis[:, 1:].repeat_interleave(N_CTX, dim=1)         # [B, K*n_ctx]
    tab_bias = _visibility_attn_bias(token_vis, prompt_learner.tab.num_heads)
    tab_layer = prompt_learner.tab.encoder.layers[0]
    tab_normed = tab_layer.norm1(raw_part_ctx)
    _, tab_attn_full = tab_layer.self_attn(tab_normed, tab_normed, tab_normed, attn_mask=tab_bias,
                                            need_weights=True, average_attn_weights=False)
    tab_attn_full = tab_attn_full.mean(dim=1)  # [B, K*n_ctx, K*n_ctx], averaged over heads
    # Aggregate each part's n_ctx context tokens into one part-to-part K x K summary (mean over
    # both the query-side and key-side n_ctx sub-blocks) -- purely for readability; TAB's own real
    # attention is over individual context tokens, not whole parts.
    Bc = tab_attn_full.shape[0]
    tab_attn = tab_attn_full.view(Bc, NUM_PARTS, N_CTX, NUM_PARTS, N_CTX).mean(dim=(2, 4)).cpu().numpy()

print("VAB attention matrix, sample 0 (rows=query part, cols=key part, parts 1..5):")
print(np.round(vab_attn[0], 3))
print()
print("TAB attention matrix, sample 0 (rows=query part, cols=key part, parts 1..5):")
print(np.round(tab_attn[0], 3))

In [ ]:
# VAB projected back into image space: for each output token i, sum_j vab_attn[i,j] * (BPAM's own
# part-j map) -- literally which pixels actually fed into that output token after VAB's mixing.
attn_part_maps = attn_pixel_probs_up[:, 1:]  # [B, K, H, W] -- drop the background channel
fig, axes = plt.subplots(NUM_ATTN_SAMPLES, 1 + NUM_PARTS,
                          figsize=(2.2 * (1 + NUM_PARTS), 2.6 * NUM_ATTN_SAMPLES))
for i in range(NUM_ATTN_SAMPLES):
    axes[i, 0].imshow(attn_raw_images[i]); axes[i, 0].axis('off')
    axes[i, 0].set_title('image' if i == 0 else '', fontsize=9)
    for out_k in range(NUM_PARTS):
        effective_map = np.tensordot(vab_attn[i, out_k], attn_part_maps[i], axes=(0, 0))  # [H, W]
        ax = axes[i, 1 + out_k]
        ax.imshow(attn_raw_images[i])
        ax.imshow(effective_map, cmap='jet', alpha=0.55, vmin=0, vmax=effective_map.max() + 1e-8)
        if i == 0:
            ax.set_title(f'VAB out {out_k + 1}', fontsize=9)
        ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Raw 5x5 matrices for VAB and TAB, side by side, sample 0 -- the only form TAB's signal can take.
fig2, axes2 = plt.subplots(1, 2, figsize=(8, 4))
im0 = axes2[0].imshow(vab_attn[0], cmap='viridis', vmin=0, vmax=1)
axes2[0].set_title('VAB attention (part tokens)')
axes2[0].set_xticks(range(NUM_PARTS)); axes2[0].set_xticklabels([f'p{k+1}' for k in range(NUM_PARTS)])
axes2[0].set_yticks(range(NUM_PARTS)); axes2[0].set_yticklabels([f'p{k+1}' for k in range(NUM_PARTS)])
plt.colorbar(im0, ax=axes2[0], fraction=0.046)

im1 = axes2[1].imshow(tab_attn[0], cmap='viridis', vmin=0, vmax=1)
axes2[1].set_title('TAB attention (prompt-context tokens)')
axes2[1].set_xticks(range(NUM_PARTS)); axes2[1].set_xticklabels([f'p{k+1}' for k in range(NUM_PARTS)])
axes2[1].set_yticks(range(NUM_PARTS)); axes2[1].set_yticklabels([f'p{k+1}' for k in range(NUM_PARTS)])
plt.colorbar(im1, ax=axes2[1], fraction=0.046)

plt.tight_layout()
plt.show()

## What this actually shows (ran with `ATTN_SEED = 3`)

**The technique works** -- both blocks return real, non-uniform attention weights, not zeros or
an error.

**VAB has a clear, if not yet ideal, pattern.** Its 5x5 matrix is far from uniform (row values
range from 0.026 to 0.533, not a flat 0.2 everywhere), but nearly every row's *largest* entry
lands on the same column (`part_2`, the shoulders/chest token) -- meaning after VAB's mixing, all
5 output tokens end up drawing heavily on the shoulders/chest region. That shows up directly in
the image-space figure: "VAB out 1" through "VAB out 5" all light up roughly the same upper-body
area, rather than each output staying visually distinct the way Stage 0's own 5 raw part maps
were (compare against the Stage 0 section above). Given this VAB was only trained for 2 epochs,
read this as "VAB hasn't yet learned to keep its 5 outputs spatially distinct from each other",
not as a final verdict -- but it's a concrete, useful thing to re-check once a real, longer Stage
1 run is available: if this pattern persists, it's a sign VAB's mixing is homogenizing the parts
more than intended.

**TAB's attention is close to flat (~0.02-0.07 everywhere, on a 0-1 scale).** Visually its matrix
reads as almost uniformly dark next to VAB's -- i.e., at this point in training it hasn't
developed a clear "which other part's prompt-context is most relevant" preference yet, unlike VAB.
Whether that's simply "needs more training" (the same explanation as VAB's own imperfect pattern)
or something specific to how TAB is used (recall it only trains during Stage 1, and is discarded
right after -- see `pcr/models/relation_blocks.py`'s own docstring) is worth revisiting once a
real, longer Stage 1 checkpoint exists.

**Directly answering the original question**: yes, TAB has a real, extractable attention signal --
but it will only ever be a part-to-part matrix like the one on the right above, never an image
overlay like VAB's, because TAB's tokens are learnable per-identity parameters that no image ever
passes through.